# Data Loading and Library Import

In [57]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.stats import boxcox

from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, precision_score, recall_score)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import xgboost as xgb


from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.under_sampling import EditedNearestNeighbours, TomekLinks

import optuna

import pickle as pkl
RANDOM_STATE = 67

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv('filtered_data.csv')

In [8]:
df.head()

,_BMI5,_BMI5CAT,_AGE_G,_AGE80,_AGE65YR,_INCOMG1,INCOME3,_SMOKER3,_RFSMOK3,SMOKDAY2,...,MENTHLTH,ADDEPEV3,LANDSEX3,SEXVAR,DIABETE4,PERSDOC3,PRIMINS2,_URBSTAT,_IMPRACE,CHILDREN
0,2249.0,2.0,6.0,78.0,2.0,9.0,99.0,4.0,1.0,NaN,...,88.0,2.0,2.0,2.0,3.0,2.0,3.0,1.0,1.0,88.0
1,2583.0,3.0,6.0,80.0,2.0,7.0,11.0,3.0,1.0,3.0,...,88.0,2.0,1.0,1.0,3.0,1.0,3.0,1.0,1.0,88.0
2,2253.0,2.0,5.0,59.0,1.0,9.0,99.0,1.0,2.0,1.0,...,88.0,2.0,1.0,1.0,3.0,3.0,1.0,1.0,1.0,88.0
3,2509.0,3.0,6.0,80.0,2.0,4.0,6.0,4.0,1.0,NaN,...,88.0,2.0,1.0,1.0,3.0,1.0,3.0,1.0,1.0,88.0
4,1977.0,2.0,4.0,47.0,1.0,2.0,3.0,4.0,1.0,NaN,...,88.0,2.0,1.0,1.0,3.0,1.0,5.0,1.0,1.0,88.0


In [9]:
df.shape

(457670, 33)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Data columns (total 33 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _BMI5     414633 non-null  float64
 1   _BMI5CAT  414633 non-null  float64
 2   _AGE_G    457670 non-null  float64
 3   _AGE80    457670 non-null  float64
 4   _AGE65YR  457670 non-null  float64
 5   _INCOMG1  457670 non-null  float64
 6   INCOME3   448401 non-null  float64
 7   _SMOKER3  457670 non-null  float64
 8   _RFSMOK3  457670 non-null  float64
 9   SMOKDAY2  167076 non-null  float64
 10  CVDINFR4  457668 non-null  float64
 11  CVDCRHD4  457667 non-null  float64
 12  ASTHMA3   457667 non-null  float64
 13  _LTASTH1  457670 non-null  float64
 14  CHCKDNY2  457664 non-null  float64
 15  MARITAL   457661 non-null  float64
 16  EDUCA     457663 non-null  float64
 17  _EDUCAG   457670 non-null  float64
 18  GENHLTH   457665 non-null  float64
 19  EXERANY2  457667 non-null  float64
 20  _TOTINDA  45767

In [11]:
chosen_columns = ['_BMI5', '_AGE_G', 'INCOME3', '_SMOKER3', 'CVDINFR4', 'CVDCRHD4',
                 'ASTHMA3', 'CHCKDNY2', 'MARITAL', 'EDUCA', 'GENHLTH', 'EXERANY2',
                 'HAVARTH4', 'MENTHLTH', 'ADDEPEV3', 'PRIMINS2', '_URBSTAT', 'CHILDREN', 'DIABETE4', 'SEXVAR']

df = df[chosen_columns]

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Data columns (total 20 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _BMI5     414633 non-null  float64
 1   _AGE_G    457670 non-null  float64
 2   INCOME3   448401 non-null  float64
 3   _SMOKER3  457670 non-null  float64
 4   CVDINFR4  457668 non-null  float64
 5   CVDCRHD4  457667 non-null  float64
 6   ASTHMA3   457667 non-null  float64
 7   CHCKDNY2  457664 non-null  float64
 8   MARITAL   457661 non-null  float64
 9   EDUCA     457663 non-null  float64
 10  GENHLTH   457665 non-null  float64
 11  EXERANY2  457667 non-null  float64
 12  HAVARTH4  457665 non-null  float64
 13  MENTHLTH  457667 non-null  float64
 14  ADDEPEV3  457665 non-null  float64
 15  PRIMINS2  457667 non-null  float64
 16  _URBSTAT  443047 non-null  float64
 17  CHILDREN  452064 non-null  float64
 18  DIABETE4  457666 non-null  float64
 19  SEXVAR    457670 non-null  float64
dtypes: float64(20)


# Data Cleaning

We're gonna be starting with simple value remapping. From the data's dictionary, we found out that there is some columns that are encoded weirdly, so we will be manually remapping those weird columns with a more proper values. Since these changes are rather independent, we will be doing it directly without splitting first since there's no data leak potential. Here are the list of changes that we made:

- Most of the binary column were mapped with 1 for yes, and 2 for no. To create a proper binary column, we've remapped them to the usual 0 for no and 1 for yes.
- Most categorical column uses 7/77 for don't know or not sure, 8/88 for 0, and 9/99 for refused, so we've swapped them with NaN values so that we can impute them with the previously stated reason above.
- Column GENHLTH's ordinality is reversed, where 1 was for excellent health while 5 was for poor health. In order to give the ordinality real meaning we've reversed| the order of the value.
- DIABETE4 has 4 categories, which were Diabetic, Pregnancy-related Diabetic, Pre-Diabetic, and Non-Diabetic. We are gonna me removing the pregnancy-related one since it is very specific and usually dissapear after pregnancy is done. And then since we are creating an early warning system, we have decided that we are going to merge Diabetic and Pre-Diabetic.
- _BMI5 was for whatever reason stored after it has been multiplied by 100. I'm assuming this happened because they might need to get rid of floating points value for the specific XPT format. To get the original value back, we will be dividing this column with 100.

In [13]:
def map_df(df):
    df['DIABETE4'] = df['DIABETE4'].map({
        1: 1, 2: np.nan, 3: 0, 4: 1, 7: np.nan, 9: np.nan
    })

    # Binary columns
    binary_cols = [
        'EXERANY2', 'CVDINFR4', 'CVDCRHD4',
        'ASTHMA3', 'CHCKDNY2', 'HAVARTH4', 'ADDEPEV3'
    ]
    binary_map = {1: 1, 2: 0, 7: np.nan, 9: np.nan}
    for col in binary_cols:
        df[col] = df[col].map(binary_map)

    # Age
    df['_AGE_G'] = df['_AGE_G'].map({1:1, 2:2, 3:3, 4:4, 5:5, 6:6})

    # Income
    df['INCOME3'] = df['INCOME3'].replace({77: np.nan, 99: np.nan})

    # General Health
    df['GENHLTH'] = df['GENHLTH'].map({1:5, 2:4, 3:3, 4:2, 5:1, 7:np.nan, 9:np.nan})

    # Education
    df['EDUCA'] = df['EDUCA'].replace({9: np.nan})

    # Sex
    df['SEXVAR'] = df['SEXVAR'].map({1:0, 2:1})

    # Urban/Rural
    df['_URBSTAT'] = df['_URBSTAT'].map({1:0, 2:1})

    # Nominal columns
    nominal_cols = ['MARITAL', 'PRIMINS2', '_SMOKER3']
    for col in nominal_cols:
        df[col] = df[col].replace({7: np.nan, 9: np.nan, 77: np.nan, 99: np.nan})

    # BMI scaling
    df['_BMI5'] = df['_BMI5'] / 100.0

    # Children
    df['CHILDREN'] = df['CHILDREN'].replace({88: 0, 99: np.nan})

    # Mental health
    df['MENTHLTH'] = df['MENTHLTH'].replace({88: 0, 77: np.nan, 99: np.nan})

    return df

In [14]:
df_mapped = map_df(df)

df_mapped

,_BMI5,_AGE_G,INCOME3,_SMOKER3,CVDINFR4,CVDCRHD4,ASTHMA3,CHCKDNY2,MARITAL,EDUCA,GENHLTH,EXERANY2,HAVARTH4,MENTHLTH,ADDEPEV3,PRIMINS2,_URBSTAT,CHILDREN,DIABETE4,SEXVAR
0,22.49,6,NaN,4.0,0.0,0.0,0.0,0.0,3.0,4.0,3.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,1
1,25.83,6,11.0,3.0,0.0,1.0,0.0,0.0,1.0,6.0,5.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0
2,22.53,5,NaN,1.0,0.0,0.0,0.0,0.0,6.0,5.0,4.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,25.09,6,6.0,4.0,0.0,0.0,0.0,0.0,1.0,6.0,5.0,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0
4,19.77,4,3.0,4.0,0.0,0.0,0.0,0.0,5.0,5.0,3.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457665,NaN,6,NaN,NaN,0.0,0.0,0.0,0.0,1.0,2.0,NaN,1.0,0.0,2.0,0.0,3.0,NaN,NaN,0.0,0
457666,20.66,6,6.0,1.0,0.0,0.0,0.0,0.0,5.0,2.0,2.0,0.0,0.0,1.0,NaN,88.0,NaN,0.0,0.0,0
457667,24.37,6,8.0,4.0,0.0,0.0,1.0,0.0,5.0,6.0,3.0,0.0,0.0,0.0,0.0,1.0,NaN,0.0,1.0,0
457668,24.41,4,10.0,3.0,0.0,0.0,0.0,0.0,1.0,5.0,5.0,1.0,0.0,0.0,0.0,2.0,NaN,0.0,0.0,0


Next, we will be handling the missing values. It has been analyzed before from EDA_Modelling that we have a clear case of MNAR/MAR missing values. Because of that, we have decided that we will not be dropping those columns since the same pattern may appear on an actual real-world case. So, we have experimented with some techniques, and the summarized technique below yielded the best result for us:

- Simple Median/Mode Imputer on column that only has <5% missing values. We are keeping it simple for these columns since small amount of static imputation won't really ruin the original distribution.
- RF Imputer + XGB Selected Features on column that has >5% missing values. We decided to do this because imputing these columns with static imputer will create a massive spike in the distribution, which may make the model not perform as well as it would've on a normally distributed data.

Since these technique has a data leak potential, we will be splitting the data first before performing it.

In [15]:
df_mapped.shape

(457670, 20)

In [16]:
df_mapped.isna().sum()

_BMI5       43037
_AGE_G          0
INCOME3     87423
_SMOKER3    32022
CVDINFR4     3115
CVDCRHD4     4578
ASTHMA3      1861
CHCKDNY2     1979
MARITAL      4222
EDUCA        2363
GENHLTH      1310
EXERANY2     1315
HAVARTH4     2573
MENTHLTH     8156
ADDEPEV3     2664
PRIMINS2    47812
_URBSTAT    14623
CHILDREN     9379
DIABETE4     4429
SEXVAR          0
dtype: int64

In [17]:
df_mapped = df_mapped.dropna(subset=['DIABETE4'])

df_train, df_test = train_test_split(df_mapped, test_size=0.2, random_state=RANDOM_STATE, stratify=df_mapped['DIABETE4'])

df_train.shape, df_test.shape

((362592, 20), (90649, 20))

In [18]:
display(df_train.isna().sum())
print()
display(df_test.isna().sum())

_BMI5       33699
_AGE_G          0
INCOME3     69153
_SMOKER3    25295
CVDINFR4     2269
CVDCRHD4     3387
ASTHMA3      1305
CHCKDNY2     1374
MARITAL      3260
EDUCA        1802
GENHLTH       974
EXERANY2      978
HAVARTH4     1801
MENTHLTH     6360
ADDEPEV3     1909
PRIMINS2    37882
_URBSTAT    11605
CHILDREN     7306
DIABETE4        0
SEXVAR          0
dtype: int64

_BMI5        8549
_AGE_G          0
INCOME3     17175
_SMOKER3     6269
CVDINFR4      632
CVDCRHD4      920
ASTHMA3       335
CHCKDNY2      372
MARITAL       803
EDUCA         438
GENHLTH       268
EXERANY2      256
HAVARTH4      468
MENTHLTH     1608
ADDEPEV3      502
PRIMINS2     9387
_URBSTAT     2891
CHILDREN     1885
DIABETE4        0
SEXVAR          0
dtype: int64

In [19]:
def predictive_impute(train_df, test_df, target_col, feature_cols, categorical_model, regression_model, is_categorical=True):
    train_df = train_df.copy()
    test_df = test_df.copy()

    known = train_df[train_df[target_col].notna()]
    unknown = train_df[train_df[target_col].isna()]

    model = categorical_model if is_categorical else regression_model
    le = LabelEncoder() if is_categorical else None

    if unknown.shape[0] > 0:
        X_known = known[feature_cols].copy()
        y_known = known[target_col]
        X_unknown = unknown[feature_cols].copy()

        # Encode categoricals as integers
        for col in X_known.columns:
            if X_known[col].dtype == 'object' or X_known[col].dtype.name == 'category':
                cats = list(X_known[col].dropna().unique())
                mapping = {k:i for i,k in enumerate(cats)}
                X_known[col] = X_known[col].map(mapping)
                X_unknown[col] = X_unknown[col].map(mapping)

        #  target encoding for xgberror
        if is_categorical:
            y_train_encoded = le.fit_transform(y_known)
        else:
            y_train_encoded = y_known

        #  TRAIN MODEL
        model.fit(X_known, y_train_encoded)

        #  PREDICT & DECODE
        preds = model.predict(X_unknown)
        if is_categorical:
            preds = le.inverse_transform(preds)

        train_df.loc[train_df[target_col].isna(), target_col] = preds

    # PREDICT MISSING IN TEST 	Val_F2 	Test_Acc 	Test_F1_Mac 	Test_F2 	Test_R0 	Test_P0 	Test_R1 	Test_P1 	TP 	TN 	FP 	FN 	Financial_Impact_USD 	Age_Impact
    test_unknown = test_df[test_df[target_col].isna()]
    if test_unknown.shape[0] > 0:
        X_test_unknown = test_unknown[feature_cols].copy()
        for col in X_test_unknown.columns:
            if X_test_unknown[col].dtype == 'object' or X_test_unknown[col].dtype.name == 'category':
                # Gunakan mapping dari train untuk konsistensi
                cats = list(known[col].dropna().unique())
                mapping = {k:i for i,k in enumerate(cats)}
                X_test_unknown[col] = X_test_unknown[col].map(mapping)

        test_preds = model.predict(X_test_unknown)
        if is_categorical:
            test_preds = le.inverse_transform(test_preds)

        test_df.loc[test_df[target_col].isna(), target_col] = test_preds

    return train_df[target_col], test_df[target_col]

def rf_impute(df_train, df_test):
    """
    Train/test-safe imputation:
    - Simple imputation (median/mode) for most columns
    - Predictive imputation for tricky columns (_BMI5, INCOME3, _SMOKER3)
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    # --- 1. Columns by type for simple imputation ---
    numeric_cols = ['CHILDREN', 'MENTHLTH']  # median only
    ordinal_mode_cols = ['_AGE_G', 'GENHLTH', 'EDUCA']  # mode only
    cat_mode_cols = [
        'MARITAL', 'SEXVAR', 'PRIMINS2',
        '_URBSTAT', 'EXERANY2', 'CVDINFR4', 'CVDCRHD4', 'ASTHMA3',
        'CHCKDNY2', 'HAVARTH4', 'ADDEPEV3'
    ]  # mode only

    # --- 2. Simple imputation ---
    # Numeric
    num_imputer = SimpleImputer(strategy='median')
    df_train[numeric_cols] = num_imputer.fit_transform(df_train[numeric_cols])
    df_test[numeric_cols] = num_imputer.transform(df_test[numeric_cols])

    # Ordinal
    ord_imputer = SimpleImputer(strategy='most_frequent')
    df_train[ordinal_mode_cols] = ord_imputer.fit_transform(df_train[ordinal_mode_cols])
    df_test[ordinal_mode_cols] = ord_imputer.transform(df_test[ordinal_mode_cols])

    # Categorical
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_train[cat_mode_cols] = cat_imputer.fit_transform(df_train[cat_mode_cols])
    df_test[cat_mode_cols] = cat_imputer.transform(df_test[cat_mode_cols])

    # --- 3. Apply predictive imputation on tricky columns ---
    categorical_model = RandomForestClassifier(random_state=RANDOM_STATE)
    regression_model = RandomForestRegressor(random_state=RANDOM_STATE)
    # INCOME3 -> categorical
    df_train['INCOME3'], df_test['INCOME3'] = predictive_impute(
        df_train, df_test, 'INCOME3',
        ['PRIMINS2', 'EDUCA', 'MARITAL', 'GENHLTH', '_AGE_G', 'EXERANY2', 'SEXVAR', '_URBSTAT'],
        categorical_model, regression_model, is_categorical=True,
    )

    # _BMI5 -> numeric
    df_train['_BMI5'], df_test['_BMI5'] = predictive_impute(
        df_train, df_test, '_BMI5',
        ['GENHLTH', 'EXERANY2', 'HAVARTH4', '_AGE_G', 'ASTHMA3', '_SMOKER3'],
        categorical_model, regression_model, is_categorical=False
    )

    # _SMOKER3 -> categorical
    df_train['_SMOKER3'], df_test['_SMOKER3'] = predictive_impute(
        df_train, df_test, '_SMOKER3',
        ['_AGE_G', 'EDUCA', 'ADDEPEV3', 'SEXVAR', 'HAVARTH4'],
        categorical_model, regression_model, is_categorical=True
    )

    return df_train, df_test

In [20]:
df_train_imputed, df_test_imputed = rf_impute(df_train, df_test)

display(df_train_imputed.isna().sum())
print()
display(df_test_imputed.isna().sum())

_BMI5       0
_AGE_G      0
INCOME3     0
_SMOKER3    0
CVDINFR4    0
CVDCRHD4    0
ASTHMA3     0
CHCKDNY2    0
MARITAL     0
EDUCA       0
GENHLTH     0
EXERANY2    0
HAVARTH4    0
MENTHLTH    0
ADDEPEV3    0
PRIMINS2    0
_URBSTAT    0
CHILDREN    0
DIABETE4    0
SEXVAR      0
dtype: int64

_BMI5       0
_AGE_G      0
INCOME3     0
_SMOKER3    0
CVDINFR4    0
CVDCRHD4    0
ASTHMA3     0
CHCKDNY2    0
MARITAL     0
EDUCA       0
GENHLTH     0
EXERANY2    0
HAVARTH4    0
MENTHLTH    0
ADDEPEV3    0
PRIMINS2    0
_URBSTAT    0
CHILDREN    0
DIABETE4    0
SEXVAR      0
dtype: int64

Nice, now that the data already has 0 missing values, let's move on to preprocessing now.

# Data Preprocessing

The data preprocessing part is mainly going to be split into two parts, which are encoding and feature engineering.

##### 1. Encoding
The encoding scheme is actually going to be really simple. We will only be applying one-hot encoding to the nominal columns. We don't really need to apply anything else since the binary and ordinal columns are either already correct or have been fixed in the mapping part earlier.

##### 2. Feature Engineering
All of our feature engineering will be based on EDA that we've made before. There are 3 feature engineering that we are gonna be applying, which are:

1. BoxCox Transform + Standard Scaling on _BMI5: The reason for this is that because _BMI5 column was initially highly skewed with a value of 1.42. After trying some transformation methods, we've found out that boxcox transform had the best result, lowering the skewness all the way to -0.01. This, however come with a trade-off where the data range is now very small, ranging from 1.3 to 1.6 only. This could be dangerous for the linear model, so we've decided to also apply standard scaling in order to compensate for this.

2. Merging CHILDREN column: The CHILDREN column had a very strange outliers, where its showing that some families could have as many as 20+ childrens, with the most being 83 children. Luckily the amount of these strange numbers are very low, so we've decided to merge them into a 4+ category based on the elbow analysis that we did in the EDA.

3. (EXPERIMENTAL) Hurdle Preprocessing MENTHLTH Column: We've decided to create a new binary 'is_menthlth' flag in order to give the model a new signal whether the sample has mental health problem or not. This was inspired by most of hurdle model articles that states that the zero data in zero-inflated data usually may have a strong signal on its own.  

In [21]:
def one_hot_encode_nominal(df_train, df_test):
    """
    One-Hot Encodes nominal columns after imputation is complete.
    Combines train and test sets temporarily to ensure the dummy columns align perfectly.
    """
    # The columns that have no mathematical order
    nominal_cols = ['MARITAL', 'PRIMINS2', '_SMOKER3']

    df_train = df_train.copy()
    df_test = df_test.copy()

    df_train['is_train_set'] = 1
    df_test['is_train_set'] = 0

    combined_df = pd.concat([df_train, df_test])

    # Apply One-Hot Encoding (drop_first=True prevents multicollinearity for LR/SVM)
    combined_encoded = pd.get_dummies(combined_df, columns=nominal_cols, drop_first=True)

    df_train_encoded = combined_encoded[combined_encoded['is_train_set'] == 1].drop(columns=['is_train_set'])
    df_test_encoded = combined_encoded[combined_encoded['is_train_set'] == 0].drop(columns=['is_train_set'])

    return df_train_encoded, df_test_encoded

In [22]:
df_train_encoded, df_test_encoded = one_hot_encode_nominal(df_train_imputed, df_test_imputed)

df_train_encoded.shape, df_test_encoded.shape

((362592, 33), (90649, 33))

In [23]:
def fe_data(df_train, df_test):
    df_tr = df_train.copy()
    df_te = df_test.copy()

    # use the same lambda to avoid data leak.
    df_tr['_BMI5'], lam = boxcox(df_tr['_BMI5'])
    df_te['_BMI5'] = boxcox(df_te['_BMI5'], lmbda=lam)

    # only fit on train
    bmi_scaler = StandardScaler()
    df_tr[['_BMI5']] = bmi_scaler.fit_transform(df_tr[['_BMI5']])
    df_te[['_BMI5']] = bmi_scaler.transform(df_te[['_BMI5']])

    # floating point error fix from the original data
    df_tr['CHILDREN'] = np.where(df_tr['CHILDREN'] < 1e-60, 0, df_tr['CHILDREN'])
    df_te['CHILDREN'] = np.where(df_te['CHILDREN'] < 1e-60, 0, df_te['CHILDREN'])

    df_tr['CHILDREN'] = df_tr['CHILDREN'].clip(upper=4)
    df_te['CHILDREN'] = df_te['CHILDREN'].clip(upper=4)

    df_tr['is_menthlth'] = (df_tr['MENTHLTH'] > 0).astype(int)
    df_te['is_menthlth'] = (df_te['MENTHLTH'] > 0).astype(int)

    return df_tr, df_te


In [24]:
df_train_fe, df_test_fe = fe_data(df_train_encoded, df_test_encoded)


display(df_train_fe.describe())
display(df_train_fe['CHILDREN'].value_counts())
df_train_fe.shape, df_test_fe.shape

,_BMI5,_AGE_G,INCOME3,CVDINFR4,CVDCRHD4,ASTHMA3,CHCKDNY2,EDUCA,GENHLTH,EXERANY2,HAVARTH4,MENTHLTH,ADDEPEV3,_URBSTAT,CHILDREN,DIABETE4,SEXVAR,is_menthlth
count,3.625920e+05,362592.000000,362592.00000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000,362592.000000
mean,9.382663e-17,4.389256,6.96511,0.058438,0.062293,0.156664,0.051656,5.028768,3.354467,0.768470,0.346342,4.325862,0.209061,0.130676,0.452820,0.170144,0.521211,0.391895
std,1.000001e+00,1.640690,2.40747,0.234569,0.241688,0.363484,0.221332,1.015916,1.048002,0.421811,0.475805,8.295363,0.406639,0.337046,0.922745,0.375760,0.499551,0.488174
min,-5.169597e+00,1.000000,1.00000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.283430e-01,3.000000,5.00000,0.000000,0.000000,0.000000,0.000000,4.000000,3.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-2.077612e-02,5.000000,7.00000,0.000000,0.000000,0.000000,0.000000,5.000000,3.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,6.040445e-01,6.000000,9.00000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
max,4.582853e+00,6.000000,11.00000,1.000000,1.000000,1.000000,1.000000,6.000000,5.000000,1.000000,1.000000,30.000000,1.000000,1.000000,4.000000,1.000000,1.000000,1.000000


CHILDREN
0.0    274678
1.0     37945
2.0     30527
3.0     12578
4.0      6864
Name: count, dtype: int64

((362592, 34), (90649, 34))

# Modelling

After we've finsihed all preprocessing, it is time for modelling. We have chosen three models to use. The variety of the model that we chose is based on the uniqueness of architecture. Because of that, we've chosen to use Logistic Regression, LinearSVC, and XGBoost. ### TODO: ADD SIMPLE REASONING WHY ARE WE USING THESE MODELS

To check for overfitting, We've also decided to apply 5-fold stratified cross validation as a benchmark to compare against test accuracy to see whether our model is overfititng or not.

In [25]:
def calculate_fin_impact(y_true, y_pred):
    """
    Menghitung dampak finansial (dalam USD) dengan faktor kepercayaan (Trust Elasticity).
    """
    cost_tp = -1265.52
    cost_tn = 0.0
    cost_fn = -28648.0
    cost_fp = -520.64

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    total_impact = (tp * cost_tp) + (tn * cost_tn) + (fp * cost_fp) + (fn * cost_fn)
    
    return total_impact


def calculate_age_impact(y_true, y_pred, age_codes):
    """
    Menghitung Life Expectancy & QALY impact dengan simulasi stokastik dan proporsional.
    """
    fn_impact_map = {1: -17.0, 2: -14.0, 3: -11.0, 4: -8.0, 5: -5.0, 6: -2.5}
    
    # Asumsi mereka terdiagnosa jauh lebih awal karena sifatnya
    # yang susah untuk dideteksi
    rescue_value = 4
    
    impacts = []

    for yt, yp, age in zip(y_true, y_pred, age_codes):
        
        # 1. Missed Diagnosis (FN)
        if yt == 1 and yp == 0:
            impacts.append(fn_impact_map[age])

        # 2. Caught in Time (TP) - Refined Logic
        elif yt == 1 and yp == 1:
            impacts.append(fn_impact_map[age] + rescue_value)

    return np.sum(impacts)



In [53]:
def train_and_evaluate(df_train, df_test, resamplers=None, models=None):
    target_col = 'DIABETE4'
    age_col = '_AGE_G'

    # Split Data Utama
    X_train_full = df_train.drop(columns=[target_col])
    y_train_full = df_train[target_col].values
    X_test_df = df_test.drop(columns=[target_col])
    y_test = df_test[target_col].values
    age_test = X_test_df[age_col].values

    # 80:20 Internal Split (Sub-train dan Validation)
    X_sub_train, X_val, y_sub_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=RANDOM_STATE
    )

    if models is None:
        models = {
            "LogisticRegression": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
            "LinearSVC": LinearSVC(max_iter=3000, random_state=RANDOM_STATE),
            "XGBoost": xgb.XGBClassifier(random_state=RANDOM_STATE)
        }

    if resamplers is None or resamplers == [None]:
        resamplers = []
        res_name = "Baseline"
    else:
        res_name = "_".join([type(r).__name__ for r in resamplers if r is not None])

    # --- Proses Resampling ---
    X_tr_res, y_tr_res = X_sub_train, y_sub_train
    for r in resamplers:
        if r is not None:
            X_tr_res, y_tr_res = r.fit_resample(X_tr_res, y_tr_res)

    X_full_res, y_full_res = X_train_full, y_train_full
    for r in resamplers:
        if r is not None:
            X_full_res, y_full_res = r.fit_resample(X_full_res, y_full_res)
                
    results_list = []

    for model_name, model in models.items():
        print(f"Processing: {model_name} with {res_name}...")

        # Training & Prediksi Validasi
        model.fit(X_tr_res, y_tr_res)
        y_pred_val = model.predict(X_val)

        # Training & Prediksi Final Test
        model.fit(X_full_res, y_full_res)
        y_pred_test = model.predict(X_test_df)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

        results_list.append({
            "Model_Resampler": f"{model_name}_{res_name}",
            
            "Val_Acc": accuracy_score(y_val, y_pred_val),
            "Val_BAcc": balanced_accuracy_score(y_val, y_pred_val),
            "Val_Sens (R1)": recall_score(y_val, y_pred_val, pos_label=1),
            "Val_Spec (R0)": recall_score(y_val, y_pred_val, pos_label=0),
            
            "Test_Acc": accuracy_score(y_test, y_pred_test),
            "Test_BAcc": balanced_accuracy_score(y_test, y_pred_test),
            "Test_Sens (R1)": recall_score(y_test, y_pred_test, pos_label=1),
            "Test_Spec (R0)": recall_score(y_test, y_pred_test, pos_label=0),
            
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "Financial_Impact_USD": calculate_fin_impact(y_test, y_pred_test),
            "Age_Impact": calculate_age_impact(y_test, y_pred_test, age_test)
        })

    return pd.DataFrame(results_list)

In [54]:
pd.options.display.float_format = '{:,.2f}'.format

## Resampling Experiments

Other than that, we are also going to be directly addressing the data imbalance problem here by comparing baseline results vs oversampled data, undersampled data, and balanced resampled data. The methods that we are going to use are SMOTE, SMOTENC, ENN, TOMEK LINKS, and a combination of whichever is the best one later.

### Baseline

Because the data is super imbalanced, my hypothesis  is that it will have decent accuracy, but a really bad F2 score since the recall for the minority class is still going to be super low

In [55]:
result_base = train_and_evaluate(df_train_fe, df_test_fe)
result_base

Processing: LogisticRegression with Baseline...


NameError: name 'balanced_accuracy_score' is not defined

My hypothesis was exactly correct.

### Oversampling

My new hypothesis is the recall for minority class should increase, but the precision might not be good yet since we are oversampling 200k+ data, which may cause too much over-generalization

#### SMOTE

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)

resamplers = [smote]

results_smote = train_and_evaluate(df_train_fe, df_test_fe, resamplers)

In [ ]:
results_smote

This yielded a very interesting result, LogReg and LinearSVC went exactly as my hypothesis, but XGBoost still struggles with minor class recall.

#### SMOTE-NC

In [ ]:
excluded_features = ['_BMI5', 'MENTHLTH', 'DIABETE4']
cat_features = [feature for feature in df_train_fe.columns if feature not in excluded_features]

smotenc = SMOTENC(random_state=RANDOM_STATE, categorical_features=cat_features)

resamplers = [smotenc]

results_smotenc = train_and_evaluate(df_train_fe, df_test_fe, resamplers)

In [ ]:
results_smotenc

### Undersampling

Now if the undersampling method that we choose is correct in reducing redundant samples, this should yield a way better class 1 precision and recall, but it may reduce class 0 recall.

#### ENN

In [ ]:
X_train = df_train_fe.drop(columns=['DIABETE4'])
y_train = df_train_fe['DIABETE4']

This yields an interesting result. There's no clear eblow that I choose to find a balanced spot between signal loss and data loss. For now I'm just gonna choose k=3 since it was the one that had the biggest spike from the previous k.

In [ ]:
enn = EditedNearestNeighbours(n_neighbors=3, n_jobs=-1)

resamplers = [enn]

result_enn = train_and_evaluate(df_train_fe, df_test_fe, resamplers)

In [ ]:
result_enn

In [ ]:
enn = EditedNearestNeighbours(n_neighbors=5, n_jobs=-1)

resamplers = [enn]


result_enn_k5 = train_and_evaluate(df_train_fe, df_test_fe, resamplers)

In [ ]:
result_enn_k5

#### Tomek Links

In [ ]:
tl = TomekLinks(sampling_strategy='majority', n_jobs=-1)

resamplers = [tl]
result_tl = train_and_evaluate(df_train_fe, df_test_fe, resamplers)

In [ ]:
result_tl

### Balanced Resampling

#### SMOTENC-ENN

Now let's try to combine the best oversampling and undersampling methods to see if they can yield a better score overall.

In [ ]:
excluded_features = ['_BMI5', 'MENTHLTH', 'DIABETE4']
cat_features = [feature for feature in df_train_fe.columns if feature not in excluded_features]

smotenc = SMOTENC(random_state=RANDOM_STATE, categorical_features=cat_features)

enn = EditedNearestNeighbours(n_neighbors=3, sampling_strategy='all', n_jobs=-1)

resamplers = [smotenc, enn]
result_balanced = train_and_evaluate(df_train_fe, df_test_fe, resamplers)

In [ ]:
result_balanced

### Final Resampling Result

In [ ]:
results = [
    result_base.assign(Strategy='Base (Imbalanced)'),
    results_smote.assign(Strategy="SMOTE"),
    results_smotenc.assign(Strategy='SMOTE-NC'),
    result_enn.assign(Strategy='ENN (k=3)'),
    result_enn_k5.assign(Strategy='ENN (k=5)'),
    result_tl.assign(Strategy='Tomek Links'),
    result_balanced.assign(Strategy='Balanced (SMOTE-NC + ENN)')
]

df_comparison = pd.concat(results, ignore_index=True)

df_final = df_comparison.sort_values(by=['Financial_Impact_USD'], ascending=False)
df_final.to_csv('result_resample.csv')

pd.options.display.float_format = '{:,.4f}'.format
df_final

## Class Weight Balancing Experiments

Judging from the financial and age impact, we can make a conclusion that F2-score aligns the best with them. So, moving forward we will be optimizing the model to reach the best F2 Score possible. Based on the F2 score, smotenc-enn resampling yielded the best results overall, so we will be continuing the experiment with this resampling method.

In [48]:
target_col = 'DIABETE4'

X_train_full = df_train_fe.drop(columns=[target_col])
y_train_full = df_train_fe[target_col]

X_test = df_test_fe.drop(columns=[target_col])
y_test = df_test_fe[target_col]

# Split Internal untuk Optuna
X_sub_train, X_val, y_sub_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=RANDOM_STATE
)

excluded_features = ['_BMI5', 'MENTHLTH', 'CHILDREN'] 
cat_feature_names = [feature for feature in X_train_full.columns if feature not in excluded_features]
# 3. Konversi nama kolom menjadi indeks numerik untuk SMOTENC
cat_indices = [X_train_full.columns.get_loc(col) for col in cat_feature_names]

smotenc = SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE)
enn = EditedNearestNeighbours(n_neighbors=3, sampling_strategy='all', n_jobs=-1)

print("Resampling Sub-Train...")
X_sub_sm, y_sub_sm = smotenc.fit_resample(X_sub_train, y_sub_train)
X_sub_res, y_sub_res = enn.fit_resample(X_sub_sm, y_sub_sm)

print("Resampling Full Train...")
X_full_sm, y_full_sm = smotenc.fit_resample(X_train_full, y_train_full)
X_full_res, y_full_res = enn.fit_resample(X_full_sm, y_full_sm)

df_train = pd.concat([X_full_res, y_full_res], axis=1)
df_sub_train = pd.concat([X_sub_res, y_sub_res], axis=1)
df_val = pd.concat([X_val, y_val], axis=1)

df_test = df_test_fe.copy() 

print("\n--- Ringkasan Distribusi Target ---")
print(f"Distribusi df_sub_train (Optuna Train) :\n{df_sub_train[target_col].value_counts(normalize=True)}")
print(f"Distribusi df_val (Optuna Eval)        :\n{df_val[target_col].value_counts(normalize=True)}")

Resampling Sub-Train...
Resampling Full Train...

--- Ringkasan Distribusi Target ---
Distribusi df_sub_train (Optuna Train) :
DIABETE4
1.0000   0.5910
0.0000   0.4090
Name: proportion, dtype: float64
Distribusi df_val (Optuna Eval)        :
DIABETE4
0.0000   0.8299
1.0000   0.1701
Name: proportion, dtype: float64


In [51]:
def objective_svc(trial):    
    w1 = trial.suggest_float('weight_class_1', 1.0, 20.0)
    class_weight_dict = {0: 1.0, 1: w1}
    
    model = LinearSVC(
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        max_iter=3000,
        dual=False
    )
    
    model.fit(X_sub_train, y_sub_train)
    
    y_pred_val = model.predict(X_val)
    
    f2 = fbeta_score(y_val, y_pred_val, beta=2)
    
    return f2

study_svc = optuna.create_study(direction="maximize", study_name="LinearSVC_F2_Optimization")

study_svc.optimize(objective_svc, n_trials=50, show_progress_bar=True)

print("\n=== Hasil Terbaik LinearSVC ===")
print(f"F2 Score Terbaik pada Validation Set : {study_svc.best_value:.4f}")
print(f"Hyperparameter Terbaik               : {study_svc.best_params}")

[I 2026-04-04 13:35:14,350] A new study created in memory with name: LinearSVC_F2_Optimization


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-04 13:35:15,583] Trial 0 finished with value: 0.5897773269186344 and parameters: {'weight_class_1': 14.297906277753992}. Best is trial 0 with value: 0.5897773269186344.
[I 2026-04-04 13:35:16,929] Trial 1 finished with value: 0.6111378828878922 and parameters: {'weight_class_1': 6.004946185713122}. Best is trial 1 with value: 0.6111378828878922.
[I 2026-04-04 13:35:18,246] Trial 2 finished with value: 0.6084460895880749 and parameters: {'weight_class_1': 5.749408751755302}. Best is trial 1 with value: 0.6111378828878922.
[I 2026-04-04 13:35:20,133] Trial 3 finished with value: 0.5726947983724617 and parameters: {'weight_class_1': 18.847005068904902}. Best is trial 1 with value: 0.6111378828878922.
[I 2026-04-04 13:35:21,476] Trial 4 finished with value: 0.46955135791948743 and parameters: {'weight_class_1': 2.582888282395751}. Best is trial 1 with value: 0.6111378828878922.
[I 2026-04-04 13:35:23,226] Trial 5 finished with value: 0.5843334190078886 and parameters: {'weight_c

In [52]:
def objective_lr(trial):    
    w1 = trial.suggest_float('weight_class_1', 1.0, 20.0)
    class_weight_dict = {0: 1.0, 1: w1}
    
    model = LogisticRegression(
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        max_iter=3000,
        solver='lbfgs'
    )
    
    model.fit(X_sub_train, y_sub_train)
    
    y_pred_val = model.predict(X_val)
    
    f2 = fbeta_score(y_val, y_pred_val, beta=2)
    
    return f2

print("Memulai Optuna Study untuk LogisticRegression...")
study_lr = optuna.create_study(direction="maximize", study_name="LogReg_F2_Optimization")

study_lr.optimize(objective_lr, n_trials=50, show_progress_bar=True)

print("\n=== Hasil Terbaik Logistic Regression ===")
print(f"F2 Score Terbaik pada Validation Set : {study_lr.best_value:.4f}")
print(f"Hyperparameter Terbaik               : {study_lr.best_params}")

[I 2026-04-04 13:37:12,137] A new study created in memory with name: LogReg_F2_Optimization


Memulai Optuna Study untuk LogisticRegression...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-04 13:37:22,503] Trial 0 finished with value: 0.597795830841873 and parameters: {'weight_class_1': 13.676766288663526}. Best is trial 0 with value: 0.597795830841873.
[I 2026-04-04 13:37:30,438] Trial 1 finished with value: 0.6013787614254904 and parameters: {'weight_class_1': 5.327473765541025}. Best is trial 1 with value: 0.6013787614254904.
[I 2026-04-04 13:37:40,074] Trial 2 finished with value: 0.5974512589298582 and parameters: {'weight_class_1': 13.813233377430068}. Best is trial 1 with value: 0.6013787614254904.
[I 2026-04-04 13:37:48,910] Trial 3 finished with value: 0.5796078507730051 and parameters: {'weight_class_1': 19.343292531972537}. Best is trial 1 with value: 0.6013787614254904.
[I 2026-04-04 13:37:57,850] Trial 4 finished with value: 0.5779181767088509 and parameters: {'weight_class_1': 19.994857262480043}. Best is trial 1 with value: 0.6013787614254904.
[I 2026-04-04 13:38:06,670] Trial 5 finished with value: 0.5802235714912238 and parameters: {'weight_cl